# Lezione 8 · Attention e Transformer, la tecnica regina

Il giro al mercato: confrontare, dividere, raccogliere. Prima per una parola sola, poi per tutte insieme, in forma di matrici.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## La pesca e la barca

In [2]:
def softmax(v):
    e = np.exp(v - np.max(v))
    return e / e.sum()

q_pesca = np.array([1, 1])
k_barca, k_ieri = np.array([2, 0]), np.array([0, 0])
gocce = softmax(np.array([q_pesca @ k_barca, q_pesca @ k_ieri]))
controlla("gocce a barca", gocce[0], 0.88)
controlla("gocce a ieri", gocce[1], 0.12)
gocce_scalate = softmax(np.array([q_pesca @ k_barca, q_pesca @ k_ieri]) / math.sqrt(2))
controlla("gocce a barca, divise per √2", gocce_scalate[0], 0.80, 0.01)

✓  gocce a barca: calcolato 0.8808, nel libro 0.88
✓  gocce a ieri: calcolato 0.1192, nel libro 0.12
✓  gocce a barca, divise per √2: calcolato 0.8044, nel libro 0.8


## Il mercato tutto insieme

Tre parole, nessuna maschera, nessuna divisione per √d_k, come nei conti del libro.

In [3]:
Q = np.array([[0, 0], [1, 0], [1, 1]])        # ieri, barca, pesca
K = np.array([[0, 0], [2, 0], [1, 1]])
V = np.array([[0, 0], [1, 0], [0.5, 0.5]])

punteggi = Q @ K.T
monete = np.apply_along_axis(softmax, 1, punteggi)
print(np.round(monete * 100))
for nome, riga, attesi in [("ieri", 0, [33, 33, 33]), ("barca", 1, [9, 66, 25]), ("pesca", 2, [6, 47, 47])]:
    for k, att in zip(["ieri", "barca", "pesca"], attesi):
        controlla(f"{nome} → {k} (%)", monete[riga][["ieri", "barca", "pesca"].index(k)] * 100, att, 0.6)

cesti = monete @ V
controlla("cesto di pesca, mare", cesti[2, 0], 0.70)
controlla("cesto di pesca, frutta", cesti[2, 1], 0.23)

scalate = np.apply_along_axis(softmax, 1, punteggi / math.sqrt(2))
for k, att in zip([0, 1, 2], [11, 45, 45]):
    controlla(f"pesca con √2, banco {k}", scalate[2, k] * 100, att, 0.6)

[[33. 33. 33.]
 [ 9. 67. 24.]
 [ 6. 47. 47.]]
✓  ieri → ieri (%): calcolato 33.33, nel libro 33
✓  ieri → barca (%): calcolato 33.33, nel libro 33
✓  ieri → pesca (%): calcolato 33.33, nel libro 33
✓  barca → ieri (%): calcolato 9.003, nel libro 9
✓  barca → barca (%): calcolato 66.52, nel libro 66
✓  barca → pesca (%): calcolato 24.47, nel libro 25
✓  pesca → ieri (%): calcolato 6.338, nel libro 6
✓  pesca → barca (%): calcolato 46.83, nel libro 47
✓  pesca → pesca (%): calcolato 46.83, nel libro 47
✓  cesto di pesca, mare: calcolato 0.7025, nel libro 0.7
✓  cesto di pesca, frutta: calcolato 0.2342, nel libro 0.23
✓  pesca con √2, banco 0: calcolato 10.84, nel libro 11
✓  pesca con √2, banco 1: calcolato 44.58, nel libro 45
✓  pesca con √2, banco 2: calcolato 44.58, nel libro 45


## Il conto del quadrato

In [4]:
for n in [1_000, 10_000, 100_000, 1_000_000]:
    print(f"{n:>9,} parole → {n*n:,} confronti")

    1,000 parole → 1,000,000 confronti
   10,000 parole → 100,000,000 confronti
  100,000 parole → 10,000,000,000 confronti
1,000,000 parole → 1,000,000,000,000 confronti


## Kata per casa

**Esercizio 1.**

In [5]:
q = np.array([1, 0]); k = np.array([[1, 0], [0, 1]]); v = np.array([10, 20])
pesi = softmax(k @ q / math.sqrt(2))
controlla("peso del primo token", pesi[0], 0.67)
controlla("uscita", pesi @ v, 13.3, 0.05)

✓  peso del primo token: calcolato 0.6698, nel libro 0.67
✓  uscita: calcolato 13.3, nel libro 13.3


**Esercizio 2.**

In [6]:
controlla("quante volte più grande", (4000**2) / (1000**2), 16, 0)

✓  quante volte più grande: calcolato 16, nel libro 16
